# 05 - Biên dịch như một bài toán MDP

Bốn notebook trước xây đủ nền để làm việc này cho đúng. Notebook này phát biểu **qubit routing**
thành một Markov decision process, hiện thực nó thành một `gymnasium.Env` thật, và đặt sẵn các mốc
so sánh - để folder `03_` chỉ việc gắn agent vào.

Kết quả bàn giao là hai file cạnh notebook:

- **`bench.py`** - harness đo đạc (notebook `04`)
- **`routing_env.py`** - môi trường + policy mốc + hàm kiểm tra

Notebook này **không** train agent nào. Lý do: một environment sai semantics hoặc một baseline yếu
làm mọi kết quả train sau đó vô giá trị, nên nó phải được kiểm tra xong trước đã.

In [ ]:
import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium.utils.env_checker import check_env
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.passes import BasicSwap, SabreSwap

import bench
import routing_env as rev

CM = CouplingMap.from_line(5)
print("gymnasium:", gym.__version__)
print("routing_env xuất ra:", [t for t in dir(rev) if not t.startswith("_") and t[0].islower()])

## 1. Phát biểu MDP

Notebook `02` cho thấy vòng lặp routing chỉ có **đúng một** chỗ cần quyết định. Đó là lý do bài toán
này khớp với RL một cách tự nhiên - không phải vì "RL giải được mọi thứ", mà vì cấu trúc của nó
đúng là một MDP:

| Thành phần MDP | Trong routing |
|---|---|
| **state** $s$ | front layer của DAG còn lại + mapping $\pi$ hiện tại |
| **action** $a$ | chọn một edge của coupling map để chèn SWAP |
| **transition** | chèn SWAP → cập nhật $\pi$ → thực thi hết cổng đã hợp lệ (bắt buộc, không phải lựa chọn) |
| **reward** $r$ | $-1$ mỗi SWAP, hoặc $-\Delta\text{depth}$, hoặc $\log$ fidelity |
| **terminal** | DAG rỗng |

Tính Markov có thoả: $(F, \pi)$ cùng với DAG còn lại xác định đầy đủ tương lai - không cần biết đã
đi đường nào tới đây. Đây là điều kiện bắt buộc để dùng RL, và nó đúng ở đây.

Một chi tiết của transition đáng nhấn: **việc thực thi cổng là bắt buộc**, không phải một action.
Nếu cho agent tự quyết định "thực thi hay không" thì action space lớn hơn mà không thêm khả năng -
hoãn một cổng đã hợp lệ không bao giờ có lợi.

In [ ]:
def sinh_mach(rng):
    """Được gọi ở mỗi reset() -> agent học trên PHÂN PHỐI mạch, không học thuộc một mạch."""
    return bench.mach_ngau_nhien(5, 10, int(rng.integers(10**6)))


env = rev.RoutingEnv(CM, sinh_mach, kieu_reward="swap")
obs, info = env.reset(seed=0)

print("action_space     :", env.action_space)
print("các edge tương ứng:", env.edges)
print("observation_space:", env.observation_space)
print("obs shape        :", obs.shape, "=", f"2 * {env.n}^2 + 1")
print("\ninfo:", {k: v for k, v in info.items() if k != "action_mask"})
print("action_mask      :", info["action_mask"])

## 2. Encode state: ba quyết định thiết kế

Encode state là chỗ dễ làm sai nhất, và cách làm quyết định agent học được hay không.

**(a) Dùng qubit vật lý, không dùng qubit logic.** Quan sát ghi "hai qubit *vật lý* $a, b$ đang cần
kề nhau", không phải "qubit logic 3 và 7 cần gặp nhau". Nhờ vậy quan sát **bất biến** với cách đánh
số qubit logic - đổi tên qubit logic không làm policy phải học lại. Đây là một dạng bất biến nhóm,
và bỏ qua nó thì agent phải học $n!$ phiên bản của cùng một tình huống.

**(b) Thêm một lớp lookahead.** Chỉ nhìn front layer là đủ Markov nhưng rất khó học: một SWAP tốt
cho hiện tại có thể tệ cho hai cổng tiếp theo. Notebook `02` cho thấy heuristic `lookahead` hơn
`basic` chính vì lý do này, nên quan sát gồm cả "extended set" - các cổng 2-qubit ngay sau front
layer.

**(c) Thêm một số đo tiến độ.** Tỉ lệ cổng đã thực thi, để agent biết mình đang ở đầu hay cuối
episode (gần cuối thì không cần đầu tư SWAP cho tương lai nữa).

Quan sát vì vậy là `[front (n×n) | lookahead (n×n) | tiến độ (1)]`.

In [ ]:
n = env.n
front = obs[:n * n].reshape(n, n)
ke_tiep = obs[n * n:2 * n * n].reshape(n, n)

print("front layer - cặp qubit VẬT LÝ đang cần kề nhau:")
print(pd.DataFrame(front.astype(int), index=[f"Q{i}" for i in range(n)],
                   columns=[f"Q{i}" for i in range(n)]).to_string())
can = [(a, b) for a in range(n) for b in range(a + 1, n) if front[a][b]]
co_san = sorted(set(tuple(sorted(e)) for e in CM.get_edges()))
print(f"\ncần kề: {can}")
print(f"có sẵn: {co_san}")
print(f"-> {[c for c in can if c not in co_san]} chưa kề nhau, nên cần SWAP.")
print(f"\nlookahead có {int(ke_tiep.sum() / 2)} cặp; tiến độ = {obs[-1]:.2f}")

## 3. Action masking

Coupling map `line(5)` có 4 edge, nhưng ở mỗi bước chỉ vài edge là **có nghĩa**: SWAP trên edge
không liên quan tới qubit nào đang bị kẹt thì chắc chắn chỉ làm mạch dài thêm.

Môi trường đưa mask qua `info["action_mask"]` nhưng **vẫn nhận** action bị mask - vì mọi RL library
đều cần `step()` không raise. Trách nhiệm dùng mask thuộc về agent (hoặc wrapper).

In [ ]:
env2 = rev.RoutingEnv(CM, sinh_mach)
obs, info = env2.reset(seed=3)
ty_le = []
buoc = 0
terminated = truncated = False
rng = np.random.default_rng(0)
while not (terminated or truncated) and buoc < 12:
    mask = info["action_mask"]
    ty_le.append(mask.sum() / len(mask))
    if buoc < 4:
        hop_le = [env2.edges[i] for i in np.flatnonzero(mask)]
        print(f"  bước {buoc}: {mask.sum()}/{len(mask)} action hợp lệ -> {hop_le}")
    a = int(rng.choice(np.flatnonzero(mask)))
    obs, r, terminated, truncated, info = env2.step(a)
    buoc += 1

print(f"\ntrung bình {np.mean(ty_le) * 100:.0f}% action là có nghĩa trên line(5) - 4 edge thì mask ít tác dụng.")

lon = CouplingMap.from_heavy_hex(7)
n_edge = len(set(tuple(sorted(e)) for e in lon.get_edges()))
print(f"\nNhưng heavy_hex(d=7) có {lon.size()} qubit và {n_edge} edge:")
print(f"  một front layer thường chỉ kẹt vài qubit, nên số action có nghĩa là một con số nhỏ")
print(f"  trong {n_edge} - lúc đó mask là bắt buộc, không có nó agent tiêu gần hết")
print("  thời gian học chỉ để tránh action vô nghĩa.")

## 4. Thiết kế reward: ba lựa chọn, không cái nào hiển nhiên đúng

Đây là quyết định quan trọng nhất của cả bài toán, vì **reward định nghĩa cái agent sẽ giỏi**.

| `kieu_reward` | Công thức | Ưu | Nhược |
|---|---|---|---|
| `"swap"` | $-1$ mỗi SWAP | đơn giản, dễ so với literature | bỏ qua depth và noise |
| `"depth"` | $-\Delta\text{depth}$ | thưởng SWAP song song được | SWAP "miễn phí" vẫn bị phạt gián tiếp |
| `"fidelity"` | $3\log(1-\epsilon_{ab})$ | gần mục tiêu thật nhất | cần `error_rate`, phụ thuộc thiết bị |

Notebook `02` đã cho bằng chứng rằng ba cái này **không cùng chiều**: `grid(3,3)` ít SWAP hơn
`ring(9)` nhưng depth cao hơn. Nên "agent giảm 10% số SWAP" không tự động nghĩa là mạch chạy tốt
hơn trên thiết bị thật.

`"fidelity"` là lựa chọn gần mục tiêu cuối cùng nhất: nó dùng error rate thật của từng edge
(notebook `01`), nên agent tự học được rằng SWAP qua edge tốt thì rẻ hơn. Hệ số 3 vì một SWAP là
3 cổng 2-qubit.

In [ ]:
backend = GenericBackendV2(num_qubits=5, coupling_map=CM.get_edges(), seed=11)
loi_cx = {e: p.error for e, p in backend.target["cx"].items()}

rows = []
for kieu in ["swap", "depth", "fidelity"]:
    e = rev.RoutingEnv(CM, sinh_mach, kieu_reward=kieu,
                       error_rate=loi_cx if kieu == "fidelity" else None)
    kq = [rev.chay_episode(e, rev.policy_greedy, seed=s) for s in range(15)]
    rows.append({
        "kieu_reward": kieu,
        "reward trung bình": round(np.mean([k["reward"] for k in kq]), 4),
        "swap": round(np.mean([k["swap"] for k in kq]), 2),
        "depth": round(np.mean([k["depth"] for k in kq]), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\n-> CÙNG một policy, ba reward khác nhau. Số swap và depth giống nhau vì policy")
print("   không đổi; điều đổi là cái một agent SẼ tối ưu nếu được train bằng reward đó.")
print(f"\nreward 'fidelity' = log của xác suất thành công: exp({rows[2]['reward trung bình']:.4f})"
      f" = {np.exp(rows[2]['reward trung bình']):.4f}")

## 5. Cổng kiểm tra: hợp lệ **và** đúng

Notebook `01` và `02` đã chỉ ra hai cái bẫy. Trong môi trường RL chúng nguy hiểm hơn, vì agent
**tối ưu hoá chính cái ta đo**: nếu chỉ đo số SWAP, một môi trường có lỗ hổng sẽ bị agent khai thác
đúng vào lỗ hổng đó.

`kiem_tra_ket_qua` kiểm tra cả hai:

1. **Hợp lệ** - `CheckMap`: mọi cổng 2-qubit nằm trên edge thật của coupling map.
2. **Đúng** - cùng unitary với mạch gốc, bù hoán vị bằng **nghịch đảo** của $\pi$.

Chạy 100 episode cho mỗi policy, yêu cầu 100% qua cả hai:

In [ ]:
env3 = rev.RoutingEnv(CM, sinh_mach, kieu_reward="swap")

rows = []
for ten, pol in [("RandomPolicy", rev.policy_ngau_nhien), ("GreedyPolicy", rev.policy_greedy)]:
    kq = [rev.chay_episode(env3, pol, seed=s) for s in range(100)]
    sw = [k["swap"] for k in kq]
    rows.append({
        "policy": ten,
        "swap mean": round(float(np.mean(sw)), 2),
        "swap std": round(float(np.std(sw)), 2),
        "xong": f"{sum(k['xong'] for k in kq)}/100",
        "hợp lệ": all(k["hop_le"] for k in kq),
        "đúng": all(k["dung"] for k in kq),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\n-> 100% hợp lệ và đúng cho cả hai policy, kể cả policy ngẫu nhiên.")
print("   Nếu môi trường cho phép sinh mạch sai thì con số này sẽ KHÁC 100%,")
print("   và đó là lúc phải sửa môi trường chứ không phải sửa agent.")

## 6. Đặt mốc: agent sẽ phải vượt những ai

So sánh phải trên **cùng tập mạch** và **cùng layout** (tầm thường, vì môi trường bắt đầu từ đó).
Đây là bảng mà folder `03_` sẽ thêm một dòng vào.

In [ ]:
rows = []
for s in range(20):
    qc = bench.mach_ngau_nhien(5, 10, s)
    e = rev.RoutingEnv(CM, lambda rng, qc=qc: qc)     # mạch cố định để so ghép cặp
    for ten, pol in [("RandomPolicy (trong env)", rev.policy_ngau_nhien),
                     ("GreedyPolicy (trong env)", rev.policy_greedy)]:
        k = rev.chay_episode(e, pol, seed=s)
        assert k["hop_le"] and k["dung"]
        rows.append({"mach": s, "cach": ten, "swap": k["swap"], "depth": k["depth"]})
    for ten, r in [("BasicSwap", BasicSwap(CM)),
                   ("SabreSwap(basic)", SabreSwap(CM, heuristic="basic", seed=s)),
                   ("SabreSwap(decay)", SabreSwap(CM, heuristic="decay", seed=s))]:
        kq = bench.do_mot_lan(qc, CM, r)
        rows.append({"mach": s, "cach": ten, "swap": kq["swap"], "depth": kq["depth"]})

df = pd.DataFrame(rows)
bang = df.groupby("cach", sort=False)[["swap", "depth"]].mean().round(2).sort_values("swap")
goc = df[df["cach"] == "SabreSwap(decay)"].set_index("mach")["swap"]
bang["kém hơn decay %"] = [
    round(((df[df["cach"] == c].set_index("mach")["swap"] - goc) / goc.replace(0, np.nan)).mean() * 100, 1)
    for c in bang.index
]
print(f"20 mạch 5 qubit trên line(5), layout tầm thường, ghép cặp từng mạch:\n")
print(bang.to_string())
print("\n-> Mục tiêu của 03_: đưa một policy học được từ khoảng 25% (GreedyPolicy) về 0%.")
print("-> Vượt được SabreSwap(decay) thì mới là kết quả mới; chỉ hơn RandomPolicy thì chưa.")

## 7. `check_env`: môi trường có hợp chuẩn gymnasium không

`gymnasium.utils.env_checker.check_env` kiểm tra những thứ mà một RL library sẽ giả định, và lỗi ở
đây thường biểu hiện thành "agent không học được" chứ không thành exception:

- `reset(seed=...)` tái lập được, và trả về `(obs, info)`
- `step()` trả về đúng 5 giá trị, `terminated` và `truncated` tách riêng (API cũ gộp thành `done`)
- `obs` thuộc đúng `observation_space`, dtype khớp
- không trả `terminated=True` **và** `truncated=True` cùng lúc

In [ ]:
env_kt = rev.RoutingEnv(CM, sinh_mach)
check_env(env_kt, skip_render_check=True)
print("check_env: PASS")

# tái lập được: cùng seed -> cùng mạch, cùng quỹ đạo
a1 = rev.chay_episode(rev.RoutingEnv(CM, sinh_mach), rev.policy_greedy, seed=7)
a2 = rev.chay_episode(rev.RoutingEnv(CM, sinh_mach), rev.policy_greedy, seed=7)
print(f"tái lập: seed 7 hai lần -> {a1['swap']} và {a2['swap']} swap, khớp = {a1['swap'] == a2['swap']}")

o1, _ = rev.RoutingEnv(CM, sinh_mach).reset(seed=7)
o2, _ = rev.RoutingEnv(CM, sinh_mach).reset(seed=7)
print("observation ban đầu khớp:", np.array_equal(o1, o2))

## 8. Hai framing còn lại

Routing không phải bài toán biên dịch duy nhất phát biểu được thành MDP. Hai cái kia, kèm baseline
đã đo ở notebook `03` và `04`:

### Synthesis as sequence (giảm T-count)

| | |
|---|---|
| **state** | unitary còn lại cần xấp xỉ (hoặc mạch Clifford+T hiện tại) |
| **action** | thêm một cổng từ gate set rời rạc / áp một luật rewrite |
| **reward** | $-1$ mỗi cổng $T$, cộng bonus khi đạt sai số $\le \epsilon$ |
| **terminal** | sai số $\le \epsilon$ |
| **baseline** | `gridsynth` 66 cổng T ở sai số $10^{-3}$; `OptimizeCliffordT` giảm 34% T-count |
| **ghi chú** | state liên tục (ma trận $2\times 2$ phức) → khác hẳn routing; reward sparse hơn nhiều |

Đây là framing của hướng AlphaTensor-Quantum. Khó hơn routing vì reward sparse và không gian state
liên tục, nhưng phần thưởng thực tế lớn hơn: T-count chi phối tài nguyên máy fault-tolerant.

### Pass ordering as bandit

| | |
|---|---|
| **state** | metric của mạch hiện tại (depth, size, số cổng 2-qubit) |
| **action** | chọn pass tiếp theo để chạy |
| **reward** | $-\Delta$(hàm cost) |
| **terminal** | hết ngân sách pass |
| **baseline** | `optimization_level` 0→3: 37 → 19 cổng cx (notebook `00`) |
| **ghi chú** | action space nhỏ nhất (~vài chục pass), **dễ thử nghiệm nhất** |

Nếu muốn có kết quả nhanh để kiểm tra toàn bộ pipeline thí nghiệm, đây là chỗ nên bắt đầu: action
space nhỏ, mỗi episode ngắn, và `bench.py` dùng được ngay không cần sửa.

## Tóm tắt

| Nội dung | Chi tiết |
|---|---|
| MDP | state $(F, \pi)$, action = chọn edge SWAP, reward âm, terminal = DAG rỗng |
| tính Markov | $(F, \pi)$ + DAG còn lại xác định đủ tương lai - điều kiện bắt buộc, và nó thoả |
| thực thi cổng | **bắt buộc**, không phải action |
| encode state | qubit **vật lý** (bất biến với đổi tên logic) + lookahead + tiến độ |
| action masking | trên `line(5)` còn ~69% action có nghĩa, nhưng `heavy_hex(7)` có 132 edge - lúc đó bắt buộc |
| reward | `swap` / `depth` / `fidelity` - **không cùng chiều**, phải chọn có ý thức |
| cổng kiểm tra | `CheckMap` **và** unitary (bù hoán vị bằng nghịch đảo $\pi$), 100% episode |
| mốc dưới | RandomPolicy - kém `SabreSwap(decay)` 266% |
| mốc giữa | GreedyPolicy (heuristic `basic`) - kém 25% |
| mốc trên | `SabreSwap(decay)` - 8.25 swap trung bình |
| hợp chuẩn | `check_env` pass, `reset(seed=)` tái lập được |

### Sang `03_` thì làm gì

1. **DQN trên `RoutingEnv`** với `kieu_reward="swap"`, line(5) - bài toán nhỏ nhất có nghĩa. Mục
   tiêu đầu tiên không phải thắng SabreSwap mà là **vượt GreedyPolicy**.
2. Dùng action masking trong mạng (đặt logit của action bị mask thành $-\infty$).
3. So bằng `bench.so_voi_baseline` trên cùng tập mạch - và nhớ nhiễu do seed ~9%.
4. Mở rộng dần: `ring(5)` → `grid(3,3)` → mạch nhiều qubit hơn. Chỗ này mới cần `torch`.

### Thử tự làm

1. Viết `policy_lookahead`: như `policy_greedy` nhưng cộng thêm $W$ lần tổng khoảng cách của lớp
   lookahead (lấy từ `obs`). Thử $W = 0.5$. Nó có thu hẹp được khoảng 25% không?
2. Đổi `sinh_mach` thành luôn sinh **một** mạch cố định. Chạy lại `policy_greedy` 100 episode -
   std bằng 0. Giải thích vì sao train trên một mạch duy nhất là cái bẫy.
3. Thêm `kieu_reward="swap_va_depth"` với reward $-1 - 0.1\,\Delta\text{depth}$. Trên bảng mốc ở
   mục 6, nó đứng đâu?
4. `RoutingEnv` hiện bắt đầu từ layout tầm thường. Thêm tham số cho phép nhận $\pi$ ban đầu từ
   `VF2Layout`. Số SWAP của `GreedyPolicy` giảm bao nhiêu? So với 16% mà `SabreLayout` đạt được ở
   notebook `02` thì thế nào?
5. Dựng môi trường cho **pass ordering** theo bảng ở mục 8: `action_space = Discrete(k)` với $k$
   pass tối ưu hoá, reward là $-\Delta$(số cổng 2-qubit). Dùng `bench.py` để so với
   `optimization_level=3`.